In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# =========================
# Cell 1
# Load dataset
# =========================

import pandas as pd

path = "/content/drive/MyDrive/Plan A/final/mimiciv_processed.csv"
df = pd.read_csv(path)

print("Shape:", df.shape)

Shape: (65366, 189)


In [7]:
# =========================
# Cell 2
# Patient-level split
# =========================

from sklearn.model_selection import train_test_split

# Unique patients
patients = df["subject_id"].unique()

train_patients, test_patients = train_test_split(
    patients, test_size=0.3, random_state=42
)

train_df = df[df["subject_id"].isin(train_patients)]
test_df = df[df["subject_id"].isin(test_patients)]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (45756, 189)
Test: (19610, 189)


In [8]:
# =========================
# Cell 3
# Define X, y
# =========================

target = "mortality"

drop_cols = ["subject_id", "hadm_id", "stay_id", "intime", "outtime"]

X_train = train_df.drop(columns=drop_cols + [target])
y_train = train_df[target]

X_test = test_df.drop(columns=drop_cols + [target])
y_test = test_df[target]

print("Features:", X_train.shape[1])

Features: 183


In [9]:
# =========================
# Cell 4
# Logistic Regression
# =========================

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred_prob_lr = lr.predict_proba(X_test)[:, 1]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
# =========================
# Cell 5
# XGBoost
# =========================

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_prob_xgb = xgb.predict_proba(X_test)[:, 1]

In [11]:
# =========================
# Cell 6
# Evaluation
# =========================

from sklearn.metrics import roc_auc_score, average_precision_score

print("Logistic Regression:")
print("AUROC:", roc_auc_score(y_test, y_pred_prob_lr))
print("AUPRC:", average_precision_score(y_test, y_pred_prob_lr))

print("\nXGBoost:")
print("AUROC:", roc_auc_score(y_test, y_pred_prob_xgb))
print("AUPRC:", average_precision_score(y_test, y_pred_prob_xgb))

Logistic Regression:
AUROC: 0.8413180629401942
AUPRC: 0.5030790731648427

XGBoost:
AUROC: 0.8866671772535468
AUPRC: 0.590628760316443


In [12]:
# =========================
# Cell 7
# Random Forest
# =========================

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_prob_rf = rf.predict_proba(X_test)[:, 1]

In [13]:
# =========================
# Cell 8
# LightGBM
# =========================

!pip install lightgbm

from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=200,
    max_depth=-1,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)

y_pred_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]

[LightGBM] [Info] Number of positive: 4942, number of negative: 40814
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013429 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8120
[LightGBM] [Info] Number of data points in the train set: 45756, number of used features: 102
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.108008 -> initscore=-2.111255
[LightGBM] [Info] Start training from score -2.111255


In [14]:
# =========================
# Cell 9
# Evaluation (All models)
# =========================

from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(name, y_true, y_pred):
    print(f"{name}:")
    print("AUROC:", roc_auc_score(y_true, y_pred))
    print("AUPRC:", average_precision_score(y_true, y_pred))
    print()

evaluate("Logistic Regression", y_test, y_pred_prob_lr)
evaluate("XGBoost", y_test, y_pred_prob_xgb)
evaluate("Random Forest", y_test, y_pred_prob_rf)
evaluate("LightGBM", y_test, y_pred_prob_lgbm)

Logistic Regression:
AUROC: 0.8413180629401942
AUPRC: 0.5030790731648427

XGBoost:
AUROC: 0.8866671772535468
AUPRC: 0.590628760316443

Random Forest:
AUROC: 0.8673156140458819
AUPRC: 0.5475364173365888

LightGBM:
AUROC: 0.8857559185351155
AUPRC: 0.5850090070532223



In [15]:
# =========================
# Cell 10
# Calibration + utility metrics
# =========================

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, precision_recall_curve, precision_score, recall_score

def expected_calibration_error(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1

    ece = 0.0
    total = len(y_true)

    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        bin_acc = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += (mask.sum() / total) * abs(bin_acc - bin_conf)

    return ece

def precision_at_recall_threshold(y_true, y_prob, target_recall=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    precision = precision[:-1]
    recall = recall[:-1]

    valid = np.where(recall >= target_recall)[0]
    if len(valid) == 0:
        return np.nan, np.nan

    idx = valid[-1]
    return precision[idx], thresholds[idx]

def precision_at_top_fraction(y_true, y_prob, fraction=0.10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    k = max(1, int(np.ceil(len(y_prob) * fraction)))
    top_idx = np.argsort(y_prob)[-k:]
    pred = np.zeros_like(y_true)
    pred[top_idx] = 1

    return precision_score(y_true, pred), recall_score(y_true, pred)

results = []

model_preds = {
    "Logistic Regression": y_pred_prob_lr,
    "XGBoost": y_pred_prob_xgb,
    "Random Forest": y_pred_prob_rf,
    "LightGBM": y_pred_prob_lgbm,
}

for name, probs in model_preds.items():
    brier = brier_score_loss(y_test, probs)
    ece = expected_calibration_error(y_test, probs, n_bins=15)
    p80, thr80 = precision_at_recall_threshold(y_test, probs, target_recall=0.80)
    p10, r10 = precision_at_top_fraction(y_test, probs, fraction=0.10)

    results.append({
        "model": name,
        "brier": brier,
        "ece_15bin": ece,
        "precision_at_80_recall": p80,
        "threshold_at_80_recall": thr80,
        "precision_at_top10pct": p10,
        "recall_at_top10pct": r10
    })

metrics_df = pd.DataFrame(results)
metrics_df

,model,brier,ece_15bin,precision_at_80_recall,threshold_at_80_recall,precision_at_top10pct,recall_at_top10pct
0,Logistic Regression,0.073338,0.008013,0.253771,0.082486,0.506884,0.463619
1,XGBoost,0.065872,0.010328,0.338328,0.099586,0.554819,0.507463
2,Random Forest,0.069670,0.013619,0.285761,0.120000,0.531871,0.486474
3,LightGBM,0.066306,0.010138,0.341289,0.105603,0.549210,0.502332


In [16]:
# =========================
# Cell 11
# Save metrics
# =========================

save_path = "/content/drive/MyDrive/Plan A/results/mimiciv_internal_metrics.csv"
metrics_df.to_csv(save_path, index=False)
print("Saved:", save_path)

Saved: /content/drive/MyDrive/Plan A/results/mimiciv_internal_metrics.csv
